In [1]:
import jax
from jax import numpy as jnp
import pennylane as qml
import numpy as np
import optax
from tqdm.auto import tqdm

In [2]:
import sys
sys.path.append('../')  # Adjust the path as necessary to import from the parent directory

from pqcqec.simulate.simulate import get_input_data, run_circuit_with_noise_model
from pqcqec.utils.jax_utils import JAXStateDataset, JAXDataLoader
from pqcqec.circuits.modify import tokenize_qiskit_circuit  
from pqcqec.circuits.generate import generate_random_circuit
from pqcqec.noise.simple_noise import PennylaneNoisyGates
from pqcqec.models.pqc_models import StateInputModelInterleavedQuaternionModel
from pqcqec.utils.quaternions_utils import quaternion_to_zxz_angles
from pqcqec.training.jax_loss_functions import jax_fidelity_loss, jax_pure_state_fidelity


In [3]:
NUM_QUBITS = 5
NUM_GATES = 50
NUM_GATE_BLOCKS = 10
NUM_PQC_BLOCKS = 1
NUM_DATA = 2500
NUM_TEST = 20
NOISE_DIST = {'x_rad': jnp.pi/30, 'z_rad': jnp.pi/30, 'delta_x': 0, 'delta_z': 0}
BATCH_SIZE = 10
EPOCHS = 3
SEED = 42

PQC_GATES = ['rz', 'rx', 'rz']  # Single-qubit rotations

In [4]:
# Given the circuit ops, gate blocks and block index, we can simply limit the number of ops to run to gate_blocks * (block_idx + 1)
# and then run the circuit as normal, interleaving the PQC gates as we go
# No, because we train EACH BLOCK, not all blocks till that block. So we need to run the circuit up to that block, not including it.



In [5]:
jax_prng_keys = jax.random.split(jax.random.PRNGKey(SEED), 3).flatten() # Split gives us (3,2) shape, flatten to (6,) 
print(f"Using Seed and JAX PRNG Keys: {SEED, jax_prng_keys}")


# Generate ideal data
ideal_train_data = get_input_data(NUM_QUBITS, NUM_DATA, seed=jax_prng_keys[0])

# Generate noise
# train_noise = JAXNoise(x_rad=jnp.pi/100, z_rad=jnp.pi/100, shape=(num_data, num_gates * 2), seed=jax_prng_keys[1])
# print(noise_dist)
if NOISE_DIST:
    noise_model = PennylaneNoisyGates(**NOISE_DIST, seed=jax_prng_keys[1])
else:
    noise_model = PennylaneNoisyGates(seed=jax_prng_keys[1])

# Create dataset and dataloader
train_dataset = JAXStateDataset(ideal_train_data)
train_dataloader = JAXDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, seed=jax_prng_keys[2])

# Generate random circuit list
qiskit_random_circuit = generate_random_circuit(
    num_qubits=NUM_QUBITS,
    num_gates=NUM_GATES,
    gate_dist=None,
    seed=SEED
)

print("Not using Uncomputation")
qiskit_uncomp_circuit = qiskit_random_circuit

uncomp_circuit_ops = tokenize_qiskit_circuit(qiskit_uncomp_circuit)

print(f"Uncomputation Circuit Ops: {uncomp_circuit_ops}")

Using Seed and JAX PRNG Keys: (42, Array([1832780943,  270669613,   64467757, 2916123636, 2465931498,
        255383827], dtype=uint32))
Not using Uncomputation
Uncomputation Circuit Ops: [('cx', [2, 4], []), ('x', [1], []), ('z', [2], []), ('z', [0], []), ('cx', [4, 1], []), ('cx', [4, 1], []), ('cz', [1, 3], []), ('x', [3], []), ('h', [2], []), ('x', [4], []), ('z', [1], []), ('h', [2], []), ('x', [0], []), ('x', [1], []), ('cx', [0, 2], []), ('h', [3], []), ('z', [2], []), ('h', [0], []), ('cz', [1, 2], []), ('x', [1], []), ('cz', [3, 4], []), ('cx', [3, 1], []), ('z', [2], []), ('x', [1], []), ('cz', [1, 2], []), ('z', [4], []), ('x', [3], []), ('x', [4], []), ('cz', [3, 2], []), ('cx', [1, 4], []), ('cz', [4, 3], []), ('cx', [0, 4], []), ('h', [0], []), ('cz', [1, 4], []), ('z', [3], []), ('h', [4], []), ('cz', [0, 3], []), ('cx', [3, 4], []), ('cz', [4, 2], []), ('h', [4], []), ('cx', [0, 4], []), ('x', [4], []), ('z', [2], []), ('z', [2], []), ('x', [0], []), ('z', [2], []), ('x

In [6]:
num_quaternion_values = 4
param_sz = (int(NUM_PQC_BLOCKS * jnp.ceil(NUM_GATES/NUM_GATE_BLOCKS)), NUM_QUBITS, num_quaternion_values)

quaternions = jnp.zeros(param_sz)  # Initialize quaternions to zeros
quaternions = quaternions.at[..., 0].set(1.0)  # Set the scalar part to 1
print(f"Initial Quaternions Shape: {quaternions.shape}")
print(f"Initial Quaternions: \n {quaternions}")

Initial Quaternions Shape: (5, 5, 4)
Initial Quaternions: 
 [[[1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]]

 [[1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]
  [1. 0. 0. 0.]]]


In [7]:
# Define optimizer
TOTAL_STEPS = int(NUM_DATA / BATCH_SIZE)
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)
RESTART_PERIOD = int(0.25 * TOTAL_STEPS)

INIT_LR = 1e-4
PEAK_LR = 1e-2
MIN_LR = 5e-4

# 1. Warmup schedule
warmup = optax.linear_schedule(
    init_value=INIT_LR,
    end_value=PEAK_LR,
    transition_steps=WARMUP_STEPS
)

# 2. Cosine decay with restarts
def cosine_with_restart_schedule(step):
    step_in_period = step % RESTART_PERIOD
    cosine = 0.5 * (1 + jnp.cos(jnp.pi * step_in_period / RESTART_PERIOD))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine

# 3. Stitch warmup + cosine
schedule = optax.join_schedules(
    schedules=[warmup, cosine_with_restart_schedule],
    boundaries=[WARMUP_STEPS]
)

# 4. Optimizer chain
optimizer = optax.chain(
    optax.clip_by_global_norm(1.0),
    optax.scale_by_adam(eps=1e-8),
    optax.add_decayed_weights(weight_decay=1e-5),
    optax.scale_by_schedule(schedule),
    optax.scale(-1.0)
)

# Training

In [8]:
def interleave_tensor_pqc_in_circuit(base_ops:list, qubits:int, blocks:int, max_blocks:int, pqc_gates:list, params:jnp.ndarray):
    """Interleave PQC operations into the base circuit."""
    interleaved_circuit = []
    # print(params.shape)
    for i, op in enumerate(base_ops):
        # print(i, op)
        interleaved_circuit.append(op)
        if (i + 1) % blocks == 0:
            k = (i + 1) // blocks - 1
            if k >= max_blocks:
                break
            
            # print(f"Interleaving PQC operations for block {k}")
            for q in range(qubits):
                for j, g in enumerate(pqc_gates):
                    # print(g, q, (k,q,j), params[k, q, j])
                    interleaved_circuit.append((g, [q], params[k, q, j].item()))
    return interleaved_circuit


# def run_circuit_ops_to_specific_block(circuit_ops, input_state, noise_model, pqc_gates, params, block_idx, num_qubits, num_gate_blocks, batched=False):
#     """Run the circuit up to a specific block index."""
#     ops_to_run = []
#     for i, op in enumerate(circuit_ops):
#         ops_to_run.append(op)
#         if (i + 1) % num_gate_blocks == 0:
#             k = (i + 1) // num_gate_blocks - 1
#             if k == block_idx:
#                 break
#             for q in range(num_qubits):
#                 for j, g in enumerate(pqc_gates):
#                     ops_to_run.append((g, [q], params[k, q, j].detach().numpy()))
    
#     return run_circuit_with_noise_model(ops_to_run, input_state, noise_model, num_qubits, batched=batched) 

In [9]:
num_blocks = quaternions.shape[0]
print(f"Number of PQC Blocks: {num_blocks}")

Number of PQC Blocks: 5


We have  O1, O2, P1, O3, O4, P2, O5, O6, P3. 
First, we train P1, by running  o1,o2,p1. 
Then, we fix p1. 


In [10]:
import copy

def run_training_interleaved_circuit(ops, pqc_gates, pqc_params, input_state, noise_model, num_qubits, batched=False):
    ops_cpy = copy.deepcopy(ops)
    for q in range(num_qubits):
        for j, g in enumerate(pqc_gates):
            ops_cpy.append((g, [q], pqc_params[q, j].item()))

    return run_circuit_with_noise_model(ops_cpy, input_state, noise_model, num_qubits, batched=batched)

In [11]:
no_noise_model = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0, seed=0)

def update_step(params, opt_state, ideal_data, circuit_ops):
    """Perform a single update step for the model parameters."""
    
    def loss_fn(p):
        angles = jax.vmap(quaternion_to_zxz_angles)(p)
        measured = run_training_interleaved_circuit(circuit_ops, PQC_GATES, angles, ideal_data, noise_model, NUM_QUBITS, batched=True)
        simulated = run_circuit_with_noise_model(circuit_ops, ideal_data, no_noise_model, num_qubits=NUM_QUBITS, batched=True)
        return jax_fidelity_loss(simulated, measured)

    loss, grads = jax.value_and_grad(loss_fn)(params)
    grads = jax.tree.map(lambda g: jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0), grads)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    new_params = optax.apply_updates(params, updates)
    new_params = jax.tree.map(lambda p: jnp.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0), new_params)

    # Fidelity after parameter update
    angles = jax.vmap(quaternion_to_zxz_angles)(new_params)
    measured = run_training_interleaved_circuit(circuit_ops, PQC_GATES, angles, ideal_data, noise_model, NUM_QUBITS, batched=True)
    simulated = run_circuit_with_noise_model(circuit_ops, ideal_data, no_noise_model, num_qubits=NUM_QUBITS, batched=True)
    fidelity = jax_pure_state_fidelity(simulated, measured)

    return opt_state, new_params, loss, fidelity

In [12]:

for block_idx in range(num_blocks):
    training_quaternions = quaternions[block_idx]
    current_pqc_params = jax.vmap(jax.vmap(quaternion_to_zxz_angles))(quaternions)
    print(f"Training Block {block_idx + 1}/{num_blocks} with initial quaternions:\n {training_quaternions}")
    print(f"Corresponding initial PQC params:\n {current_pqc_params}")

    interleaved_circuit_ops = interleave_tensor_pqc_in_circuit(
        uncomp_circuit_ops, 
        NUM_QUBITS, 
        NUM_GATE_BLOCKS, 
        block_idx,  
        PQC_GATES, 
        current_pqc_params
    )

    print(f"Interleaved Circuit Ops up to Block {block_idx}:\n {interleaved_circuit_ops}")

    opt_state = optimizer.init(training_quaternions)
    for e in range(EPOCHS):
        print(f"Epoch {e + 1}/{EPOCHS}")
        data_iterator = tqdm(train_dataloader, desc="Training", total=len(train_dataloader), leave=False, unit='batch')
        
        # Initialize lists to track metrics for this epoch
        epoch_fidelities = []
        epoch_losses = []

        for i, batch in enumerate(data_iterator):

            # ideal_data = batch  # Assuming the first element is the ideal data
            # print(f'Batch Shape: {batch}')
            ideal_data = batch[0]  # Assuming the first element is the ideal data
            # print(f'Ideal Data Shape: {ideal_data.shape}')
            # print(f'Ideal Data \n: {ideal_data}')

            opt_state, params, loss, fidelity = update_step(training_quaternions, opt_state, ideal_data, interleaved_circuit_ops)
            
            training_quaternions = params

            # Track metrics
            epoch_fidelities.append(float(fidelity))
            epoch_losses.append(float(loss))

            current_lr = schedule(i)

            data_iterator.set_postfix_str(f"Fidelity (Ideal, Measured): {fidelity:.4e}, Loss: {loss:.4e}, LR: {current_lr:.4e}")

        print(f"Epoch {e + 1} Summary: Avg Fidelity: {np.mean(epoch_fidelities):.4e}, Avg Loss: {np.mean(epoch_losses):.4e}")
        
    print(f"Trained Quaternions for Block {block_idx}:\n {training_quaternions}")
    print(f"Corresponding trained PQC params:\n {jax.vmap(quaternion_to_zxz_angles)(training_quaternions)}")
    quaternions = quaternions.at[block_idx].set(training_quaternions)



Training Block 1/5 with initial quaternions:
 [[1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]
 [1. 0. 0. 0.]]
Corresponding initial PQC params:
 [[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]
Interleaved Circuit Ops up to Block 0:
 [('cx', [2, 4], []), ('x', [1], []), ('z', [2], []), ('z', [0], []), ('cx', [4, 1], []), ('cx', [4, 1], []), ('cz', [1, 3], []), ('x', [3], []), ('h', [2], []), ('x', [4], [])]
Epoch 1/3


Training:   0%|          | 0/250 [00:00<?, ?batch/s]

Epoch 1 Summary: Avg Fidelity: 8.7161e-01, Avg Loss: 1.2839e-01
Epoch 2/3


Training:   0%|          | 0/250 [00:00<?, ?batch/s]

Epoch 2 Summary: Avg Fidelity: 8.7157e-01, Avg Loss: 1.2843e-01
Epoch 3/3


...

Epoch 1 Summary: Avg Fidelity: 4.0789e-01, Avg Loss: 5.9211e-01
Epoch 2/3


Training:   0%|          | 0/250 [00:00<?, ?batch/s]

Epoch 2 Summary: Avg Fidelity: 4.0756e-01, Avg Loss: 5.9244e-01
Epoch 3/3


Training:   0%|          | 0/250 [00:00<?, ?batch/s]

Epoch 3 Summary: Avg Fidelity: 4.0764e-01, Avg Loss: 5.9236e-01
Trained Quaternions for Block 4:
 [[0.9999595 0.        0.        0.       ]
 [0.9999595 0.        0.        0.       ]
 [0.9999595 0.        0.        0.       ]
 [0.9999595 0.        0.        0.       ]
 [0.9999595 0.        0.        0.       ]]
Corresponding trained PQC params:
 [[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [13]:
final_circuit_ops = interleave_tensor_pqc_in_circuit(
    uncomp_circuit_ops, 
    NUM_QUBITS, 
    NUM_GATE_BLOCKS, 
    num_blocks,  
    PQC_GATES, 
    jax.vmap(jax.vmap(quaternion_to_zxz_angles))(quaternions)
)

print(f"Final Circuit Ops:\n {final_circuit_ops}")

Final Circuit Ops:
 [('cx', [2, 4], []), ('x', [1], []), ('z', [2], []), ('z', [0], []), ('cx', [4, 1], []), ('cx', [4, 1], []), ('cz', [1, 3], []), ('x', [3], []), ('h', [2], []), ('x', [4], []), ('rz', [0], 0.0), ('rx', [0], 0.0), ('rz', [0], 0.0), ('rz', [1], 0.0), ('rx', [1], 0.0), ('rz', [1], 0.0), ('rz', [2], 0.0), ('rx', [2], 0.0), ('rz', [2], 0.0), ('rz', [3], 0.0), ('rx', [3], 0.0), ('rz', [3], 0.0), ('rz', [4], 0.0), ('rx', [4], 0.0), ('rz', [4], 0.0), ('z', [1], []), ('h', [2], []), ('x', [0], []), ('x', [1], []), ('cx', [0, 2], []), ('h', [3], []), ('z', [2], []), ('h', [0], []), ('cz', [1, 2], []), ('x', [1], []), ('rz', [0], 0.0), ('rx', [0], 0.0), ('rz', [0], 0.0), ('rz', [1], 0.0), ('rx', [1], 0.0), ('rz', [1], 0.0), ('rz', [2], 0.0), ('rx', [2], 0.0), ('rz', [2], 0.0), ('rz', [3], 0.0), ('rx', [3], 0.0), ('rz', [3], 0.0), ('rz', [4], 0.0), ('rx', [4], 0.0), ('rz', [4], 0.0), ('cz', [3, 4], []), ('cx', [3, 1], []), ('z', [2], []), ('x', [1], []), ('cz', [1, 2], []), ('z